# Day 9 · Spatial Feature Extraction with CNNs

**Objective:** Build a shallow CNN (`Conv2d`, `MaxPool2d`, `Linear`) to classify leaf images as healthy or diseased.

## 1. Download the dataset

We use the New Plant Diseases dataset from Kaggle. Original folder names are converted into two labels: `0 = healthy`, `1 = diseased`.

In [ ]:
!pip -q install kaggle
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!mkdir -p /content/plant_data
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/plant_data --unzip

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_dirs = list(Path('/content/plant_data').rglob('train'))
valid_dirs = list(Path('/content/plant_data').rglob('valid'))
if not train_dirs:
    raise FileNotFoundError("No 'train' folder found after extracting the Kaggle dataset.")
train_dir = train_dirs[0]
valid_dir = valid_dirs[0] if valid_dirs else None
print('Train folder:', train_dir)
print('Validation folder:', valid_dir)

image_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])
base_train = datasets.ImageFolder(train_dir, transform=image_transform)

class BinaryLeafDataset(Dataset):
    def __init__(self, image_folder):
        self.data = image_folder
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        image, original_label = self.data[index]
        class_name = self.data.classes[original_label].lower()
        label = 0 if 'healthy' in class_name else 1
        return image, torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(BinaryLeafDataset(base_train), batch_size=32, shuffle=True)
print('Training images:', len(base_train))
print('Labels: 0 = healthy, 1 = diseased')

## 2. Build the shallow CNN

Convolution layers learn local patterns such as edges, spots, and textures. Max pooling reduces feature-map size. The linear layer converts extracted features into two class scores.

In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Linear(32 * 32 * 32, 2)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(torch.flatten(x, 1))

model = ShallowCNN().to(device)
print(model)

## 3. Train the CNN

Cross-entropy compares the two output scores with the correct class. Backpropagation calculates gradients, and Adam updates the model weights.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    print(f'Epoch {epoch+1}/5 | Loss: {total_loss/total:.4f} | Accuracy: {correct/total:.4f}')

### Day 9 · Key takeaway

A CNN learns spatial image features with convolution, reduces their dimensions with pooling, and uses a linear layer to classify the image.